# Übungen – Block 10: Rollen und Strukturierung

Ausgangspunkt ist ein Playbook aus Block 9.

## Musterlösung

## Übung 1: Rollenstruktur

Ordnen Sie Tasks, Handler, Templates, statische Dateien und Defaultwerte den passenden Rollenverzeichnissen zu.

```text
tasks -> tasks/main.yml
handlers -> handlers/main.yml
Templates -> templates/
statische Dateien -> files/
überschreibbare Standardwerte -> defaults/main.yml
```

## Übung 2: Rolle erzeugen

Erzeugen Sie eine Rolle `web_config` und untersuchen Sie die Struktur.

```bash
ansible-galaxy role init roles/web_config
tree roles/web_config
```

## Übung 3: Tasks überführen

Verschieben Sie vorhandene Tasks in `tasks/main.yml`.

`roles/web_config/tasks/main.yml`:

```yaml
- name: Verzeichnis vorhanden
  ansible.builtin.file:
    path: "{{ app_dir }}"
    state: directory

- name: Konfiguration vorhanden
  ansible.builtin.template:
    src: app.conf.j2
    dest: "{{ app_dir }}/app.conf"
  notify: Restart demo
```

## Übung 4: Template und Handler

Verschieben Sie Template und Handler in die Rolle.

`roles/web_config/handlers/main.yml`:

```yaml
- name: Restart demo
  ansible.builtin.debug:
    msg: "Dienstaktion ausgelöst"
```

Template nach `roles/web_config/templates/app.conf.j2`.

## Übung 5: Defaults

Lagern Sie `service_port` und einen Pfad nach `defaults/main.yml` aus.

`roles/web_config/defaults/main.yml`:

```yaml
service_port: 8080
app_dir: /tmp/app
```

## Übung 6: Rolle verwenden

Rufen Sie die Rolle aus einem schlanken Playbook auf.

```yaml
- name: Web-Konfiguration
  hosts: web
  roles:
    - web_config
```

## Abschlussübung

Wenden Sie dieselbe Rolle mit unterschiedlichen Variablen auf zwei Hostgruppen an.

```yaml
- name: Web
  hosts: web
  vars:
    service_port: 8080
  roles:
    - web_config

- name: Test
  hosts: test
  vars:
    service_port: 8081
  roles:
    - web_config
```

## Erläuterungen und Prüfkriterien

### Übung 1 – Rollenverzeichnisse
`tasks/` enthält auszuführende Tasks, `handlers/` benachrichtigte Tasks, `templates/` Jinja2-Dateien, `files/` statische Dateien und `defaults/main.yml` niedrig priorisierte Standardwerte. Die Verzeichnisse übernehmen unterschiedliche Aufgaben und sollten nicht austauschbar verwendet werden.

### Übung 2 – Rolle initialisieren
`ansible-galaxy role init roles/web_config` erzeugt eine Struktur. Die Existenz der Verzeichnisse ist noch keine funktionierende Rolle; mindestens `tasks/main.yml` und gegebenenfalls weitere referenzierte Dateien müssen sinnvolle Inhalte haben.

### Übung 3 und 4 – Migration
Tasks in `roles/web_config/tasks/main.yml` benötigen keine äußere `tasks:`-Klammer. Templates liegen relativ zum Rollenverzeichnis unter `templates/`, Handler unter `handlers/main.yml`. Der Name in `notify` muss zum Handler passen.

### Übung 5 – Defaults
`service_port` und `app_dir` in `defaults/main.yml` liefern sinnvolle Standardwerte. Sie sollen durch Play-, Gruppen- oder Hostvariablen überschrieben werden können, ohne die Rolle selbst zu verändern.

### Übung 6 – Rolle verwenden
Ein schlankes Playbook definiert `hosts` und `roles`; die Details liegen in der Rolle. Der Rollenname muss auflösbar sein, beispielsweise über `roles/` im Projekt oder einen konfigurierten `roles_path`.

### Abschlussübung – Wiederverwendung
Dasselbe Rollenverzeichnis wird für zwei Hostgruppen genutzt, aber mit unterschiedlichen Variablenwerten. Prüfen Sie die erzeugten Konfigurationsdateien pro Hostgruppe und den zweiten Lauf auf unnötige Änderungen. Zwei kopierte Rollen wären keine Wiederverwendung.